# MovieLens plot extraction

Build an auditable plot dataset from MovieLens 32M, Wikidata, and Wikipedia.
A film is linked through its IMDb identifier. The extractor checks the Wikidata
identity of the article after redirects and reads an explicit plot section from
a fixed revision. It keeps the original language and skips unverified leads.

Start with the **pilot**. Agree on the experiment population with the project
partner before choosing **experiments** or **catalog**. Change the configuration
in the setup cell and run the notebook from the top.

- **Pilot** uses the saved V4 movie IDs when available. Otherwise it recreates
  the original sample from the MovieLens archive with seed 42. A completed run
  writes plots_pilot.parquet and plots_pilot_coverage.json.
- **Experiments** reads an explicit CSV or Parquet with a movieId column.
  **Catalog** selects the whole MovieLens catalog. A completed run in either
  scope writes plots.parquet and plots_coverage.json in the project data folder.
- On Colab the data directory defaults to Drive under movierec/data. Locally
  it defaults to the data directory under the current working directory.
  MOVIEREC_DATA_DIR overrides either default. Put ml-32m.zip under data/raw or
  supply the extracted movies.csv and links.csv under data/raw/ml-32m.
  The notebook uses NumPy, Polars, Requests, and Beautiful Soup.
- Use one runtime per scope. Progress is stored in scope-specific Parquet
  batches under plots_checkpoints. Rerun the processing and export cells to
  continue. After a hard stop, at most the current unsaved checkpoint interval
  is repeated. A completed record is reused, while a transient error remains
  eligible for retry. Increasing the per-run budget keeps the same checkpoint.
  If Wikidata's batch API reports persistent replica lag during the pilot,
  the notebook can read those known entities via the Linked Data Interface.
- The export contains one row per selected movie. Filter status == "ok" for
  plot embeddings. The remaining rows keep a null plot and a reason for the
  genre/tag fallback. The output directory matches movierec.paths.data_dir()
  under the standard project setup.


In [1]:
# ============================================================
# Set up the extraction run
# ============================================================

import hashlib
import json
import math
import os
import re
import shutil
import time
import unicodedata
from collections import defaultdict
from datetime import UTC, datetime
from email.utils import parsedate_to_datetime
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory
from urllib.parse import quote, unquote, urlparse
from zipfile import ZipFile

import numpy as np
import polars as pl
import requests
from bs4 import BeautifulSoup

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    default_data_dir = Path("/content/drive/MyDrive/movierec/data")
else:
    default_data_dir = Path.cwd() / "data"

DATA_DIR = Path(os.environ.get("MOVIEREC_DATA_DIR", default_data_dir))
ARCHIVE_PATH = DATA_DIR / "raw" / "ml-32m.zip"
EXTRACTED_DIR = DATA_DIR / "raw" / "ml-32m"
SCOPE = "pilot"  # Use "experiments" or "catalog" after agreeing on the population
EXPERIMENT_MOVIES = DATA_DIR / "experiment_movies.parquet"  # Requires movieId
RATINGS_PATH = None  # Optional CSV or Parquet, such as a training-only split
USE_ARCHIVE_RATINGS = True  # Use the MovieLens archive for descriptive coverage
MAX_MOVIES_PER_RUN = 100
CHECKPOINT_INTERVAL = 5
MAPPING_BATCH_SIZE = 40
MAX_EDITIONS = 5
MIN_PLOT_CHARS = 150
PIPELINE_VERSION = 1  # Increment when extraction or selection rules change

WIKIDATA_QUERY = "https://query.wikidata.org/sparql"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WIKIDATA_ENTITY_DATA = "https://www.wikidata.org/wiki/Special:EntityData"
WIKIPEDIA_HOST = re.compile(r"([a-z0-9]+(?:-[a-z0-9]+)*)\.wikipedia\.org")
IMDB_ID = re.compile(r"(?:tt)?([0-9]+)")
WIKIDATA_ID = re.compile(r"Q[1-9][0-9]*")
LANGUAGE_PRIORITY = ("en", "it", "fr", "es", "de", "pt", "nl", "ru", "pl", "ja", "zh")
COMMON_HEADINGS = {"plot", "plot summary", "synopsis", "storyline", "premise"}
LOCAL_HEADINGS = {
    "an": {"argumento", "trama"},
    "ca": {"argument", "sinopsi", "trama"},
    "cy": {"crynodeb", "crynodeb o'r plot"},
    "de": {"handlung", "filminhalt", "inhaltsangabe"},
    "en": {"story", "story summary", "plot overview"},
    "es": {"argumento", "sinopsis", "trama", "resumen de la trama"},
    "et": {"sisu", "süžee"},
    "fi": {"juoni", "juonikuvaus"},
    "fr": {"synopsis détaillé", "résumé", "résumé détaillé", "intrigue"},
    "ht": {"rezime", "senopsis"},
    "it": {"trama", "sinossi", "intreccio"},
    "ja": {"あらすじ", "ストーリー", "物語"},
    "nl": {"verhaal", "verhaallijn", "inhoud"},
    "pl": {"fabuła", "opis fabuły"},
    "pt": {"enredo", "sinopse", "trama"},
    "ru": {"сюжет", "краткое содержание", "содержание фильма"},
    "sr": {"радња", "садржај", "radnja", "sadržaj"},
    "sv": {"handling", "handlingen", "intrig"},
    "uk": {"сюжет", "зміст"},
    "ur": {"کہانی", "خلاصہ", "پلاٹ"},
    "zh": {"劇情", "剧情", "故事", "劇情簡介", "剧情简介"},
}
RESULT_SCHEMA = {
    "movieId": pl.Int64,
    "title": pl.String,
    "imdbId": pl.String,
    "plot": pl.String,
    "status": pl.String,
    "quality": pl.String,
    "wikidata_id": pl.String,
    "plot_language": pl.String,
    "plot_wikipedia_url": pl.String,
    "plot_revision_id": pl.Int64,
    "plot_revision_url": pl.String,
    "plot_section": pl.String,
    "plot_chars": pl.Int64,
    "detail": pl.String,
}
RETRYABLE_STATUS = "retryable_error"

Mounted at /content/drive


## Review of the recovered multilingual pilot plots

The executed Colab pilot reported **73 candidate texts out of 100**, including
11 additions over V3. The review reconstructed each addition from the public
Wikipedia revision and matched the saved character count. The article's
Wikidata IMDb claim matched the MovieLens ID for all 11. Eight extracts passed
the content check. Two include cast annotations and one is non-narrative.
The essay film is excluded at the verified identity level because its content
is non-narrative. The two other decisions apply to the reviewed text, so the
extractor can still try an edition without cast annotations.

The saved V4 CSV, when available, is checked against the reviewed revision URL
and text hash. The historical 73% is a candidate count. The new run reports its
own coverage and separates manually reviewed texts from automatic candidates.

| MovieLens ID | Revision | IMDb ID | Section | Decision | Review note |
| --- | --- | --- | --- | --- | --- |
| 4309 | [Petits Frères](https://fr.wikipedia.org/w/index.php?oldid=220251004) | tt0184819 | Synopsis (fr) | Accepted | The runaway and her missing dog drive the story. |
| 73744 | [If You Love](https://fi.wikipedia.org/w/index.php?oldid=23002181) | tt1380130 | Juoni (fi) | Accepted | The accident, amnesia, and relationship are narrative events. |
| 107147 | [Last Winter](https://fr.wikipedia.org/w/index.php?oldid=233007047) | tt1810798 | Synopsis (fr) | Accepted | A short synopsis of the farmer and the fire. |
| 126652 | [Raven the Little Rascal](https://fr.wikipedia.org/w/index.php?oldid=222249346) | tt1695770 | Synopsis (fr) | Accepted | The damaged dam and search for help are plot events. |
| 138364 | [Una donna per la vita](https://it.wikipedia.org/w/index.php?oldid=145344216) | tt2166285 | Trama (it) | Accepted | The accident and relationships follow the film's story. |
| 145288 | [A Deadly Encounter](https://fr.wikipedia.org/w/index.php?oldid=241058875) | tt0424939 | Synopsis (fr) | Accepted | The calls and break-in belong to the story. |
| 154266 | [Critique de la séparation](https://fr.wikipedia.org/w/index.php?oldid=238668934) | tt0481821 | Synopsis (fr) | Excluded for content | Describes montage and interpretation and quotes a separate work. |
| 177169 | [Men in the City](https://nl.wikipedia.org/w/index.php?oldid=70499912) | tt2513536 | Verhaal (nl) | Held for cleanup | The narrative repeatedly includes actor names in parentheses. |
| 210311 | [Naked Weekend](https://fr.wikipedia.org/w/index.php?oldid=239918636) | tt0071499 | Synopsis (fr) | Held for cleanup | The narrative embeds cast names and actor credits. |
| 215815 | [Strongman Ferdinand](https://it.wikipedia.org/w/index.php?oldid=147810010) | tt0075266 | Trama (it) | Accepted | The security job and its consequences form the plot. |
| 216505 | [That Old Dream That Moves](https://it.wikipedia.org/w/index.php?oldid=145366109) | tt0256666 | Trama (it) | Accepted | The factory closure and relationships form the plot. |


In [2]:
# ============================================================
# Record the multilingual pilot review
# ============================================================

PILOT_REVIEW = [
    {
        "movieId": 4309,
        "language": "fr",
        "revision": 220251004,
        "wikidata_id": "Q16028797",
        "section": "Synopsis",
        "revision_url": "https://fr.wikipedia.org/w/index.php?oldid=220251004",
        "decision": "accepted",
        "digest": "81a4e3e6ef05530bd03ff6db7a0cb8655e2f9a22dd6f2821439818fd98f0a947",
    },
    {
        "movieId": 73744,
        "language": "fi",
        "revision": 23002181,
        "wikidata_id": "Q11867050",
        "section": "Juoni",
        "revision_url": "https://fi.wikipedia.org/w/index.php?oldid=23002181",
        "decision": "accepted",
        "digest": "16cf42feae1474f94c32ec3bd9641aebadfe6e078391a2d5a60c07f34d11dc4e",
    },
    {
        "movieId": 107147,
        "language": "fr",
        "revision": 233007047,
        "wikidata_id": "Q3203568",
        "section": "Synopsis",
        "revision_url": "https://fr.wikipedia.org/w/index.php?oldid=233007047",
        "decision": "accepted",
        "digest": "06a464b6a49514d6d4800bf58a1986194f32dbfe95dd2ea446cf77d1749d9964",
    },
    {
        "movieId": 126652,
        "language": "fr",
        "revision": 222249346,
        "wikidata_id": "Q15990829",
        "section": "Synopsis",
        "revision_url": "https://fr.wikipedia.org/w/index.php?oldid=222249346",
        "decision": "accepted",
        "digest": "9dc8643af468db88b738a313dc711dd49a617173465df867da3e906685c4e043",
    },
    {
        "movieId": 138364,
        "language": "it",
        "revision": 145344216,
        "wikidata_id": "Q4004216",
        "section": "Trama",
        "revision_url": "https://it.wikipedia.org/w/index.php?oldid=145344216",
        "decision": "accepted",
        "digest": "927b4425350dd9233053f66a291b5f15e255bc945b024f5d27a90483b1b7aef6",
    },
    {
        "movieId": 145288,
        "language": "fr",
        "revision": 241058875,
        "wikidata_id": "Q1354494",
        "section": "Synopsis",
        "revision_url": "https://fr.wikipedia.org/w/index.php?oldid=241058875",
        "decision": "accepted",
        "digest": "fbba3d4310867e36d8e75ea9fba28195538e760829604552854f3d79ef1768df",
    },
    {
        "movieId": 154266,
        "language": "fr",
        "revision": 238668934,
        "wikidata_id": "Q3002978",
        "section": "Synopsis",
        "revision_url": "https://fr.wikipedia.org/w/index.php?oldid=238668934",
        "decision": "excluded_non_narrative",
        "digest": "e03e8cdb2d4275381bc74da0cc7d468c6ba18c70581bdb0b120a88c3f745205f",
    },
    {
        "movieId": 177169,
        "language": "nl",
        "revision": 70499912,
        "wikidata_id": "Q15879065",
        "section": "Verhaal",
        "revision_url": "https://nl.wikipedia.org/w/index.php?oldid=70499912",
        "decision": "excluded_cast_annotations",
        "digest": "7c8f50dd2d66d73bd908445b057552ef5837eb9357da066a6cf351c4277152ca",
    },
    {
        "movieId": 210311,
        "language": "fr",
        "revision": 239918636,
        "wikidata_id": "Q5862071",
        "section": "Synopsis",
        "revision_url": "https://fr.wikipedia.org/w/index.php?oldid=239918636",
        "decision": "excluded_cast_annotations",
        "digest": "3d0820461084353d05f4c07e608cbb9f7e0c8ac5e3d3e93e9a7a21748bf5fccb",
    },
    {
        "movieId": 215815,
        "language": "it",
        "revision": 147810010,
        "wikidata_id": "Q13459679",
        "section": "Trama",
        "revision_url": "https://it.wikipedia.org/w/index.php?oldid=147810010",
        "decision": "accepted",
        "digest": "05a92227692aa57cbd66191a2807bf6358d099d0d4c840fc7a99881e1ad425b0",
    },
    {
        "movieId": 216505,
        "language": "it",
        "revision": 145366109,
        "wikidata_id": "Q2943638",
        "section": "Trama",
        "revision_url": "https://it.wikipedia.org/w/index.php?oldid=145366109",
        "decision": "accepted",
        "digest": "df87c9cf195879b924e30ac59ff8e07a76ddda4a575b03512ded47cf0976059c",
    },
]


def plot_digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


review_by_source = {
    (row["movieId"], row["language"], row["revision"], row["section"]): row for row in PILOT_REVIEW
}
review_by_text = {(row["movieId"], row["language"], row["digest"]): row for row in PILOT_REVIEW}
non_narrative_films = {
    (row["movieId"], row["wikidata_id"]): row
    for row in PILOT_REVIEW
    if row["decision"] == "excluded_non_narrative"
}
pilot_v4_csv = DATA_DIR / "plots_pilot" / "plots_pilot_100_v4.csv"
if pilot_v4_csv.is_file():
    prior = {
        row["movieId"]: row
        for row in pl.read_csv(
            pilot_v4_csv, columns=["movieId", "plot", "plot_revision_url"]
        ).iter_rows(named=True)
    }
    matched = sum(
        bool(
            (saved := prior.get(row["movieId"]))
            and saved["plot"]
            and saved["plot_revision_url"] == row["revision_url"]
            and plot_digest(saved["plot"]) == row["digest"]
        )
        for row in PILOT_REVIEW
    )
    print(f"Saved V4 CSV extracts matching the review: {matched}/{len(PILOT_REVIEW)}")
else:
    print("Pilot review uses public revisions reconstructed from the saved Colab output.")

Saved V4 CSV extracts matching the review: 11/11


In [3]:
# ============================================================
# Load MovieLens and select the scope
# ============================================================


def normalize_imdb(value):
    match = IMDB_ID.fullmatch(value.strip()) if value else None
    return f"tt{match[1].zfill(7)}" if match and int(match[1]) else None


def read_movielens_csv(name, **options):
    local = EXTRACTED_DIR / name
    if local.is_file():
        return pl.read_csv(local, **options)
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(f"Expected {local} or {ARCHIVE_PATH}.")
    with ZipFile(ARCHIVE_PATH) as archive, archive.open(f"ml-32m/{name}") as source:
        return pl.read_csv(source.read(), **options)


def load_catalog():
    movies = read_movielens_csv("movies.csv", columns=["movieId", "title"])
    links = read_movielens_csv(
        "links.csv",
        columns=["movieId", "imdbId"],
        schema_overrides={"imdbId": pl.String},
    )
    return movies.join(links, on="movieId", how="left", validate="1:1").with_columns(
        pl.col("imdbId").map_elements(normalize_imdb, return_dtype=pl.String)
    )


def select_movies(catalog):
    if SCOPE == "catalog":
        return catalog.sort("movieId")
    if SCOPE == "pilot":
        # Use the IDs from the original pilot when its V4 export is available.
        pilot_csv = DATA_DIR / "plots_pilot" / "plots_pilot_100_v4.csv"
        if not pilot_csv.is_file():
            eligible = catalog.filter(pl.col("imdbId").is_not_null()).sort("movieId")
            indices = np.random.RandomState(42).choice(
                eligible.height, size=min(100, eligible.height), replace=False
            )
            return eligible[indices.tolist()].sort("movieId")
        selection = pl.read_csv(pilot_csv, columns=["movieId"])
    elif SCOPE == "experiments":
        if EXPERIMENT_MOVIES.suffix.lower() == ".parquet":
            selection = pl.read_parquet(EXPERIMENT_MOVIES, columns=["movieId"])
        else:
            selection = pl.read_csv(EXPERIMENT_MOVIES, columns=["movieId"])
    else:
        raise ValueError("SCOPE must be pilot, experiments, or catalog.")

    selection = selection.with_columns(
        pl.col("movieId").cast(pl.String).str.strip_chars().cast(pl.Int64)
    ).unique()
    unknown = selection.join(catalog, on="movieId", how="anti")
    if unknown.height:
        raise ValueError(f"Selection contains unknown or missing MovieLens IDs: {unknown.head(5)}")
    return catalog.join(selection, on="movieId", how="semi").sort("movieId")


catalog = load_catalog()
selected = select_movies(catalog)
if selected.is_empty():
    raise ValueError("The selected population is empty.")
if min(MAX_MOVIES_PER_RUN, CHECKPOINT_INTERVAL, MAPPING_BATCH_SIZE, MAX_EDITIONS) < 1:
    raise ValueError("Batch limits must be positive.")
if MAPPING_BATCH_SIZE > 50:
    raise ValueError("Wikidata entity batches must contain at most 50 IDs.")
print(f"Scope: {SCOPE} | selected: {selected.height:,}/{catalog.height:,} MovieLens films")

Scope: pilot | selected: 100/87,585 MovieLens films


In [ ]:
# ============================================================
# Resolve IMDb identities and retrieve Wikipedia sections
# ============================================================


class RemoteError(RuntimeError):
    """A service response could not be used and should be retried in a later run."""


class ReplicaLagError(RemoteError):
    """The Action API declined the request because its replicas lagged."""


class MissingArticle(RuntimeError):
    """The article or revision is unavailable."""


wiki_session = requests.Session()
wiki_session.headers["User-Agent"] = (
    "MovieRecResearch/0.1 (https://github.com/Cxsee/AI-in-Industry-project)"
)


def retry_delay(response, default):
    header = response.headers.get("Retry-After") if response is not None else None
    if not header:
        return default
    try:
        seconds = int(header)
    except ValueError:
        try:
            seconds = (parsedate_to_datetime(header) - datetime.now(UTC)).total_seconds()
        except (TypeError, ValueError):
            return default
    return max(default, seconds)


def request_json(url, params, *, mediawiki=True):
    params = {"format": "json", "maxlag": 5, **params} if mediawiki else params
    for attempt in range(3):
        response = None
        error_type = RemoteError
        try:
            response = wiki_session.get(url, params=params, timeout=(10, 60), allow_redirects=False)
            time.sleep(1)
            if not 200 <= response.status_code < 300:
                response.raise_for_status()
                raise RemoteError(f"Unexpected HTTP {response.status_code} from {url}")
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            if response is not None and response.status_code not in {429, 500, 502, 503, 504, 200}:
                raise RemoteError(str(exc)) from exc
            message = str(exc)
        else:
            error = payload.get("error")
            if not error:
                return payload
            code = error.get("code", "")
            message = error.get("info", code)
            if code in {"missingtitle", "missingrev", "nosuchrevid", "nosuchsection"}:
                raise MissingArticle(message)
            if code == "maxlag":
                error_type = ReplicaLagError
            elif code not in {"ratelimited", "readonly"}:
                raise RemoteError(message)
        if attempt < 2:
            delay = retry_delay(response, 5 * 2**attempt)
            if delay > 120:
                raise RemoteError(f"{message}; service requested a wait of {delay:.0f} seconds")
            time.sleep(delay)
    raise error_type(message)


def imdb_matches(imdb_ids):
    values = " ".join(f'"{imdb_id}"' for imdb_id in sorted(set(imdb_ids)))
    if not values:
        return {}
    query = f"""
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
        SELECT DISTINCT ?imdb ?item WHERE {{
            VALUES ?imdb {{ {values} }}
            ?item wdt:P345 ?imdb .
        }}
    """
    response = request_json(WIKIDATA_QUERY, {"query": query, "format": "json"}, mediawiki=False)
    matches = defaultdict(set)
    for match in response["results"]["bindings"]:
        qid = match["item"]["value"].rsplit("/", 1)[-1]
        if WIKIDATA_ID.fullmatch(qid):
            matches[match["imdb"]["value"]].add(qid)
    return matches


def wikidata_entities(ids):
    if not ids:
        return {}
    try:
        return request_json(
            WIKIDATA_API,
            {
                "action": "wbgetentities",
                "ids": "|".join(sorted(ids)),
                "props": "claims|sitelinks/urls",
            },
        )["entities"]
    except ReplicaLagError:
        if SCOPE != "pilot":
            raise

    print("Wikidata batch API is lagging. Reading pilot entities via the Linked Data Interface.")
    return {
        qid: request_json(f"{WIKIDATA_ENTITY_DATA}/{qid}.json", {}, mediawiki=False)[
            "entities"
        ].get(qid, {})
        for qid in sorted(ids)
    }


def entity_imdb_ids(entity):
    return {
        claim["mainsnak"].get("datavalue", {}).get("value")
        for claim in entity.get("claims", {}).get("P345", [])
        if claim.get("rank") != "deprecated"
    }


def wikipedia_links(entity):
    pages = set()
    for sitelink in entity.get("sitelinks", {}).values():
        url = sitelink.get("url", "")
        match = WIKIPEDIA_HOST.fullmatch(urlparse(url).netloc)
        if match:
            pages.add((match[1], url))
    rank = {language: index for index, language in enumerate(LANGUAGE_PRIORITY)}
    return sorted(pages, key=lambda page: (rank.get(page[0], len(rank)), *page))


def normalize_heading(heading):
    heading = unicodedata.normalize("NFKC", heading).casefold().replace("’", "'")
    return " ".join(heading.split()).strip(" ::")


def extract_prose(html):
    soup = BeautifulSoup(html, "html.parser")
    for node in soup.select(
        "sup, table, .mw-editsection, .thumb, figure, .hatnote, .navbox, "
        ".reference, .reflist, .noprint, .shortdescription, .ambox, style, script"
    ):
        node.decompose()
    paragraphs = []
    seen_heading = False
    for node in soup.find_all(["p", "h1", "h2", "h3", "h4", "h5", "h6"]):
        if node.name == "p":
            paragraph = node.get_text(" ", strip=True)
            if paragraph:
                paragraphs.append(paragraph)
        elif seen_heading or paragraphs:
            # Subsections can contain analyses or other material outside the synopsis.
            break
        else:
            seen_heading = True
    return re.sub(r"\s+([,.;:!?])", r"\1", " ".join(" ".join(paragraphs).split()))


def article_sections(language, url, qid):
    page_url = urlparse(url)
    if (
        page_url.scheme not in {"https", "http"}
        or page_url.netloc != f"{language}.wikipedia.org"
        or not page_url.path.startswith("/wiki/")
    ):
        raise MissingArticle("Unexpected Wikipedia article URL")
    api = f"https://{page_url.netloc}/w/api.php"
    title = unquote(page_url.path.removeprefix("/wiki/")).replace("_", " ")
    query = request_json(
        api,
        {
            "action": "query",
            "titles": title,
            "redirects": 1,
            "prop": "pageprops|revisions",
            "ppprop": "wikibase_item|disambiguation",
            "rvprop": "ids",
            "formatversion": 2,
        },
    )
    page = query["query"]["pages"][0]
    if "missing" in page or "invalid" in page:
        raise MissingArticle(title)
    props = page.get("pageprops", {})
    if props.get("wikibase_item") != qid or "disambiguation" in props:
        return None
    revision = page["revisions"][0]["revid"]
    parsed = request_json(api, {"action": "parse", "oldid": revision, "prop": "tocdata"})["parse"]
    sections = parsed["tocdata"].get("sections", [])
    canonical_url = f"https://{page_url.netloc}/wiki/" + quote(page["title"].replace(" ", "_"))
    return api, revision, canonical_url, sections

In [5]:
# ============================================================
# Extract one reliable candidate per film
# ============================================================


def result(movie, status, **fields):
    return {**movie, "status": status, **fields}


def extract_from_page(movie, qid, language, url):
    article = article_sections(language, url, qid)
    if article is None:
        return None, "article_identity_mismatch"
    api, revision, canonical_url, sections = article
    headings = COMMON_HEADINGS | LOCAL_HEADINGS[language]
    outcome = "no_plot_section"
    for section in sections:
        index = section.get("index", "")
        heading = BeautifulSoup(section["line"], "html.parser").get_text(" ", strip=True)
        if not str(index).isdecimal() or normalize_heading(heading) not in headings:
            continue
        review = review_by_source.get((movie["movieId"], language, revision, heading))
        if review and review["decision"] != "accepted":
            outcome = review["decision"]
            continue
        parsed = request_json(
            api,
            {
                "action": "parse",
                "oldid": revision,
                "prop": "text",
                "section": index,
            },
        )["parse"]
        plot = extract_prose(parsed["text"]["*"])
        if len(plot) < MIN_PLOT_CHARS:
            if outcome == "no_plot_section":
                outcome = "short_plot" if plot else "empty_plot_section"
            continue
        review = review_by_text.get((movie["movieId"], language, plot_digest(plot)))
        if review and review["decision"] != "accepted":
            outcome = review["decision"]
            continue
        quality = "manual_accepted" if review else "automatic_candidate"
        return {
            "plot": plot,
            "quality": quality,
            "plot_language": language,
            "plot_wikipedia_url": canonical_url,
            "plot_revision_id": revision,
            "plot_revision_url": api.removesuffix("/api.php") + f"/index.php?oldid={revision}",
            "plot_section": heading,
            "plot_chars": len(plot),
        }, "ok"
    return None, outcome


def extract_movie(movie, matches, entities):
    imdb_id = movie["imdbId"]
    if imdb_id is None:
        return result(movie, "missing_imdb_id")
    ids = matches.get(imdb_id, set())
    if len(ids) != 1:
        status = "ambiguous_wikidata_match" if ids else "no_wikidata_match"
        return result(movie, status, detail=", ".join(sorted(ids)) or None)
    (qid,) = ids
    entity = entities.get(qid, {})
    if imdb_id not in entity_imdb_ids(entity):
        return result(
            movie, RETRYABLE_STATUS, wikidata_id=qid, detail="Wikidata IMDb claim changed"
        )
    review = non_narrative_films.get((movie["movieId"], qid))
    if review:
        return result(movie, review["decision"], wikidata_id=qid, detail=review["revision_url"])
    pages = wikipedia_links(entity)
    if not pages:
        return result(movie, "no_wikipedia_sitelink", wikidata_id=qid)
    supported = [(language, url) for language, url in pages if language in LOCAL_HEADINGS]
    if not supported:
        return result(movie, "unsupported_wikipedia_language", wikidata_id=qid)

    outcomes = []
    attempts = []
    for language, url in supported[:MAX_EDITIONS]:
        try:
            fields, status = extract_from_page(movie, qid, language, url)
        except MissingArticle as exc:
            fields, status = None, "article_unavailable"
            attempts.append(f"{language}: {exc}")
        except RemoteError as exc:
            fields, status = None, RETRYABLE_STATUS
            attempts.append(f"{language}: {exc}")
        if fields:
            return result(movie, "ok", wikidata_id=qid, **fields)
        outcomes.append(status)
        attempts.append(f"{language} {url}: {status}")

    if len(supported) > MAX_EDITIONS:
        outcomes.append("edition_limit_reached")
    priority = [
        RETRYABLE_STATUS,
        "excluded_non_narrative",
        "excluded_cast_annotations",
        "edition_limit_reached",
        "short_plot",
        "empty_plot_section",
        "no_plot_section",
        "article_identity_mismatch",
        "article_unavailable",
    ]
    status = next(outcome for outcome in priority if outcome in outcomes)
    return result(movie, status, wikidata_id=qid, detail=" | ".join(attempts))

In [ ]:
# ============================================================
# Process pending batches and checkpoint progress
# ============================================================


def save_atomically(destination, write):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=destination.parent, suffix=".part", delete=False) as file:
        temporary = Path(file.name)
    try:
        write(temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)


def run_fingerprint(population):
    spec = {
        "scope": SCOPE,
        "movies": population.rows(),
        "mapping_batch_size": MAPPING_BATCH_SIZE,
        "minimum_plot_chars": MIN_PLOT_CHARS,
        "maximum_editions": MAX_EDITIONS,
        "language_priority": LANGUAGE_PRIORITY,
        "common_headings": sorted(COMMON_HEADINGS),
        "local_headings": {lang: sorted(headings) for lang, headings in LOCAL_HEADINGS.items()},
        "review": PILOT_REVIEW,
        "version": PIPELINE_VERSION,
    }
    return hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]


def checkpoint_batches(population, root):
    for offset in range(0, population.height, MAPPING_BATCH_SIZE):
        batch = population.slice(offset, MAPPING_BATCH_SIZE)
        yield batch, root / f"batch-{offset // MAPPING_BATCH_SIZE:05d}.parquet"


def read_checkpoint(path, batch):
    if not path.is_file():
        return {}
    frame = pl.read_parquet(path)
    if frame.schema != RESULT_SCHEMA or frame["movieId"].n_unique() != frame.height:
        raise ValueError(f"Invalid checkpoint schema or duplicate movie IDs: {path}")
    if frame.join(batch, on=["movieId", "title", "imdbId"], how="anti", nulls_equal=True).height:
        raise ValueError(f"Checkpoint contains records outside its batch: {path}")
    return {row["movieId"]: row for row in frame.iter_rows(named=True)}


def write_checkpoint(path, rows):
    frame = pl.DataFrame(list(rows.values()), schema=RESULT_SCHEMA)
    save_atomically(path, frame.write_parquet)


def process_pending(population, root):
    remaining = MAX_MOVIES_PER_RUN
    for batch, path in checkpoint_batches(population, root):
        existing = read_checkpoint(path, batch)
        completed = {
            movie_id for movie_id, row in existing.items() if row["status"] != RETRYABLE_STATUS
        }
        pending = [
            movie for movie in batch.iter_rows(named=True) if movie["movieId"] not in completed
        ][:remaining]
        if not pending:
            continue
        try:
            matches = imdb_matches(movie["imdbId"] for movie in pending if movie["imdbId"])
            entities = wikidata_entities(
                {next(iter(ids)) for ids in matches.values() if len(ids) == 1}
            )
        except RemoteError as exc:
            for movie in pending:
                existing[movie["movieId"]] = result(
                    movie, RETRYABLE_STATUS, detail=f"Wikidata: {exc}"
                )
            write_checkpoint(path, existing)
            print(f"Stopped after a Wikidata failure; rerun to retry. {exc}")
            return

        unsaved = 0
        consecutive_errors = 0
        try:
            for movie in pending:
                row = extract_movie(movie, matches, entities)
                existing[movie["movieId"]] = row
                unsaved += 1
                consecutive_errors = (
                    consecutive_errors + 1 if row["status"] == RETRYABLE_STATUS else 0
                )
                if unsaved >= CHECKPOINT_INTERVAL:
                    write_checkpoint(path, existing)
                    unsaved = 0
                if consecutive_errors >= 3:
                    print("Stopped after three consecutive retryable results.")
                    return
        finally:
            if unsaved:
                write_checkpoint(path, existing)
        remaining -= len(pending)
        print(f"Batch {path.stem}: {len(pending)} attempted")
        if remaining == 0:
            break


checkpoint_root = DATA_DIR / "plots_checkpoints" / f"{SCOPE}-{run_fingerprint(selected)}"
print(f"Checkpoint directory: {checkpoint_root}")
process_pending(selected, checkpoint_root)

In [ ]:
# ============================================================
# Verify the dataset and report coverage
# ============================================================


def checkpoint_results(population, root):
    results = [
        row
        for batch, path in checkpoint_batches(population, root)
        for row in read_checkpoint(path, batch).values()
    ]
    return pl.DataFrame(results, schema=RESULT_SCHEMA).sort("movieId")


def verify_dataset(plots, population):
    if not plots.select("movieId", "title", "imdbId").equals(population.sort("movieId")):
        raise ValueError("The parquet must have exactly one row per selected MovieLens ID.")
    published = plots.filter(pl.col("status") == "ok")
    required = [
        "plot",
        "quality",
        "wikidata_id",
        "plot_language",
        "plot_wikipedia_url",
        "plot_revision_id",
        "plot_revision_url",
        "plot_section",
        "plot_chars",
    ]
    invalid = published.filter(
        pl.any_horizontal(pl.col(name).is_null() for name in required)
        | (pl.col("plot").str.len_chars() != pl.col("plot_chars"))
        | (pl.col("plot_chars") < MIN_PLOT_CHARS)
    )
    unexpected = plots.filter((pl.col("status") != "ok") & pl.col("plot").is_not_null())
    if invalid.height or unexpected.height:
        raise ValueError("Extracted text, status, and provenance do not reconcile.")


def ratings_counts(path, population):
    if path.suffix.lower() == ".parquet":
        ratings = pl.scan_parquet(path)
    else:
        ratings = pl.scan_csv(path)
    names = ratings.collect_schema().names()
    id_column = "movieId" if "movieId" in names else "movie_id"
    if id_column not in names:
        raise ValueError("Ratings must have a movieId or movie_id column.")
    return (
        ratings.select(pl.col(id_column).cast(pl.Int64).alias("movieId"))
        .join(population.select("movieId").lazy(), on="movieId", how="semi")
        .group_by("movieId")
        .len(name="rating_count")
        .collect()
    )


def popularity_groups(plots, population):
    path = (
        Path(RATINGS_PATH)
        if RATINGS_PATH
        else next(
            (
                candidate
                for candidate in (DATA_DIR / "ratings.parquet", EXTRACTED_DIR / "ratings.csv")
                if candidate.is_file()
            ),
            None,
        )
    )
    if path is not None:
        counts = ratings_counts(path, population)
        source = str(path)
    elif USE_ARCHIVE_RATINGS and ARCHIVE_PATH.is_file():
        with TemporaryDirectory() as folder:
            local = Path(folder) / "ratings.csv"
            with ZipFile(ARCHIVE_PATH) as archive:
                if "ml-32m/ratings.csv" not in archive.namelist():
                    return None
                with archive.open("ml-32m/ratings.csv") as inp, local.open("wb") as out:
                    shutil.copyfileobj(inp, out)
            counts = ratings_counts(local, population)
        source = f"{ARCHIVE_PATH}::ml-32m/ratings.csv"
    else:
        return None

    counts = counts.sort(["rating_count", "movieId"], descending=[True, False])
    top_ids = counts.head(math.ceil(counts.height / 5))["movieId"].to_list()
    joined = plots.join(counts, on="movieId", how="left").with_columns(
        pl.when(pl.col("rating_count").is_null())
        .then(pl.lit("unrated"))
        .when(pl.col("movieId").is_in(top_ids))
        .then(pl.lit("top_20_percent_rated"))
        .otherwise(pl.lit("other_rated"))
        .alias("group")
    )
    groups = (
        joined.group_by("group")
        .agg(
            pl.len().alias("movies"),
            (pl.col("status") == "ok").sum().alias("with_plot"),
        )
        .with_columns((pl.col("with_plot") / pl.col("movies")).alias("coverage"))
        .sort("group")
    )
    return {"ratings_source": source, "groups": groups.to_dicts()}


def export_plots(plots, population, root):
    finished = plots.filter(pl.col("status") != RETRYABLE_STATUS).height
    print(f"Completed: {finished:,}/{population.height:,}")
    print(plots.group_by("status").len().sort("status"))
    if finished != population.height:
        print("Rerun the processing cell to resume before exporting this scope.")
        return

    verify_dataset(plots, population)
    stem = "plots_pilot" if SCOPE == "pilot" else "plots"
    output = DATA_DIR / f"{stem}.parquet"
    coverage_path = DATA_DIR / f"{stem}_coverage.json"
    usable = plots.filter(pl.col("status") == "ok")
    report = {
        "scope": SCOPE,
        "checkpoint": root.name,
        "selected_movies": population.height,
        "movies_with_plot": usable.height,
        "coverage": usable.height / population.height,
        "quality_counts": usable.group_by("quality").len().sort("quality").to_dicts(),
        "status_counts": plots.group_by("status").len().sort("status").to_dicts(),
        "popularity": popularity_groups(plots, population),
    }

    def write_verified(path):
        plots.write_parquet(path)
        if not pl.read_parquet(path).equals(plots):
            raise ValueError("The parquet readback differs from the verified dataset.")

    save_atomically(output, write_verified)
    save_atomically(
        coverage_path,
        lambda path: path.write_text(
            json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        ),
    )
    print(f"Coverage: {report['coverage']:.1%} ({usable.height:,}/{population.height:,})")
    print(f"Saved {output} and {coverage_path}")


plots = checkpoint_results(selected, checkpoint_root)
export_plots(plots, selected, checkpoint_root)

## Dataset handoff and limits

In 04_content_based.ipynb the file can be loaded after the standard setup with:

```python
import polars as pl
from movierec import paths

plots = pl.read_parquet(paths.data_dir() / "plots.parquet")
plot_texts = plots.filter(pl.col("status") == "ok").select("movieId", "plot", "plot_language")
```

Left-join the plots by movieId to preserve the genre/tag fallback. Use an
embedding model that supports the source languages if the experiment includes
multilingual texts. The revision URL identifies the source text for attribution.

The coverage denominator is the configured population. The popularity report
uses the top 20% of rated movies in that population, the other rated movies,
and unrated movies. Supply the training ratings via RATINGS_PATH for a
training-only popularity measure. The MovieLens archive provides a descriptive
fallback. It is not used to choose the movies or the plots.

Automatic acceptance is based on the verified film identity, heading, cleaned
prose, and minimum length. Semantic quality is verified only for the reviewed
pilot revisions. Coverage can be limited by ambiguous IMDb mappings, unavailable
articles, language support, the edition limit, and missing or short sections.
